# Reconhecimento dos integrantes do grupo com CNN

Uma Rede Neural Convolucional **feita à mão, só com NumPy** (sem TensorFlow, Keras ou PyTorch) que recebe a foto de um
rosto e diz qual integrante do grupo ela é.

Este notebook roda o mesmo pipeline de `python -m src.treinar`, uma etapa por célula, **importando as funções de
`src/`**: o código é um só, e a documentação do site mostra cada função linha a linha.

## 1. Configuração

Rode o notebook a partir da pasta do repositório (ou de `notebooks/`). No Google Colab, clone o repositório e envie
as fotos para `dataset/<nome>/` antes de continuar.

In [ ]:
# No Colab, descomente:
# !git clone <url-do-repositorio> && %cd cnn-reconhecimento-facial && pip install -r requirements.txt
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                                    # as funções de src/ esperam a raiz do repositório
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score

from src.config import DATASET_DIR, IMG_SIZE, EPOCAS, SEED
from src.dados import ler_imagem, listar_dataset
from src.preprocessamento import preprocessar
from src.divisao import dividir_dados
from src.aumento import aumentar
from src.modelo import iniciar_pesos, prever, resumo_camadas
from src.metricas import calcular_metricas
from src.reconhecimento import grad_cam
from src.treinar import treinar_rede

gerador = np.random.default_rng(SEED)                 # semente fixa: o resultado se repete a cada execução

## 2. Leitura das fotos

Cada subpasta de `dataset/` é um integrante; as fotos são lidas em ordem alfabética (a ordem em que foram tiradas).

In [ ]:
classes, caminhos, rotulos = listar_dataset(DATASET_DIR)
for numero, nome in enumerate(classes):
    print(f"{numero} = {nome}: {np.sum(rotulos == numero)} fotos")

## 3. Pré-processamento

Tons de cinza → detecção do rosto (Haar, OpenCV) → recorte com margem → 64×64 → **CLAHE** (contraste local) →
suavização gaussiana → máscara oval → escala 0-1. Fotos sem rosto detectado ficam de fora.

In [ ]:
_, etapas = preprocessar(ler_imagem(caminhos[0]), mostrar_etapas=True)
titulos = ["cinza", "1. recorte", "2. 64x64", "3. CLAHE", "4. suavizada", "5. máscara oval"]
fig, eixos = plt.subplots(1, len(etapas), figsize=(3 * len(etapas), 3))
for ax, img, titulo in zip(eixos, etapas, titulos):
    ax.imshow(img, cmap="gray"); ax.set_title(titulo); ax.axis("off")
plt.show()

In [ ]:
X, usadas = [], []
for i, caminho in enumerate(caminhos):
    rosto = preprocessar(ler_imagem(caminho))
    if rosto is not None:
        X.append(rosto)
        usadas.append(i)
X, y = np.array(X), rotulos[usadas]
print(f"{len(y)} de {len(rotulos)} fotos com rosto detectado. X: {X.shape}")

## 4. Divisão em treino, validação e teste

Por **blocos de fotos seguidas** (fotos vizinhas são quase iguais e não podem cair uma no treino e outra no teste).

In [ ]:
idx_treino, idx_val, idx_teste = dividir_dados(y, classes)
X_treino, y_treino = X[idx_treino], y[idx_treino]
X_val, y_val = X[idx_val], y[idx_val]
X_teste, y_teste = X[idx_teste], y[idx_teste]
print("treino:", len(y_treino), " validação:", len(y_val), " teste:", len(y_teste))

## 5. Aumento de dados

Feito dentro do laço de treino, a cada lote: rotação, zoom, deslocamento, espelho, brilho, contraste e ruído sorteados.

In [ ]:
fig, eixos = plt.subplots(1, 7, figsize=(14, 2.4))
eixos[0].imshow(X_treino[0][:, :, 0], cmap="gray", vmin=0, vmax=1); eixos[0].set_title("original")
for ax in eixos[1:]:
    ax.imshow(aumentar(X_treino[0], gerador)[:, :, 0], cmap="gray", vmin=0, vmax=1)
for ax in eixos:
    ax.axis("off")
plt.show()

## 6. A CNN (feita à mão em `src/modelo.py`)

3 blocos (convolução 3×3 + ReLU + max pooling 2×2) → dropout → densa 64 + ReLU → densa (1 por integrante) → softmax.

In [ ]:
params = iniciar_pesos(len(classes), gerador)
for camada in resumo_camadas(params):
    print(f"{camada['nome']:>8}  {camada['tipo']:<22} saída {camada['saida']}  {camada['parametros']} pesos")

## 7. Treinamento

Laço de épocas e lotes: aumenta → propaga → entropia cruzada → retropropaga → Adam. Parada antecipada pela validação.

In [ ]:
historico = treinar_rede(params, X_treino, y_treino, X_val, y_val, gerador)
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))
for ax, chave, titulo in [(eixos[0], "loss", "Erro"), (eixos[1], "accuracy", "Acurácia")]:
    ax.plot(historico[chave], label="treino"); ax.plot(historico["val_" + chave], label="validação")
    ax.set_title(titulo); ax.set_xlabel("época"); ax.legend()
plt.show()

## 8. Avaliação no teste

In [ ]:
y_previsto = prever(params, X_teste).argmax(axis=1)
matriz = confusion_matrix(y_teste, y_previsto, labels=range(len(classes)))
m = calcular_metricas(matriz)
print("Matriz de confusão (linhas = real, colunas = previsto):")
print(matriz)
for i, nome in enumerate(classes):
    print(f"{nome:>12}: precisão {m['precisao'][i]:.3f}  recall {m['recall'][i]:.3f}  F1 {m['f1'][i]:.3f}"
          f"  especificidade {m['especificidade'][i]:.3f}")
for nome, Xc, yc in [("treino", X_treino, y_treino), ("validação", X_val, y_val), ("teste", X_teste, y_teste)]:
    print(f"Acurácia em {nome}: {accuracy_score(yc, prever(params, Xc).argmax(axis=1)):.3f}")

## 9. Onde a rede olhou (Grad-CAM)

In [ ]:
fig, eixos = plt.subplots(1, len(classes), figsize=(3 * len(classes), 3), squeeze=False)
for c, ax in enumerate(eixos[0]):
    img = X_teste[y_teste == c][0]
    calor, previsto = grad_cam(params, img)
    ax.imshow(img[:, :, 0], cmap="gray"); ax.imshow(calor, cmap="jet", alpha=0.45)
    ax.set_title(f"real: {classes[c]}\nprevisto: {classes[previsto]}"); ax.axis("off")
plt.show()